In [0]:
from pyspark.sql import functions as F

CATALOGO = "mvp_engenharia_de_dados_puc_rio"
SCHEMA = "dados_tse"

tabelas_bronze = [
    "bronze_consulta_cand_2022",
    "bronze_consulta_cand_2026",
    "bronze_consulta_cand_complementar_2022",
    "bronze_consulta_cand_complementar_2026",
]

print("========== CONTROLE DE ENTRADA PARA A SILVER ==========")

for tabela in tabelas_bronze:
    nome = f"{CATALOGO}.{SCHEMA}.{tabela}"
    df = spark.table(nome)
    print(f"{tabela}: {df.count()} registros | {len(df.columns)} colunas")

In [0]:
# ============================================================
# SILVER — Etapa 1: recorte do escopo de negócio
# Deputado Estadual — São Paulo — 2022 x 2026
# ============================================================

TABELA_2022 = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "bronze_consulta_cand_2022"
)

TABELA_2026 = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "bronze_consulta_cand_2026"
)

# Fontes principais da Bronze
bronze_2022 = spark.table(TABELA_2022)
bronze_2026 = spark.table(TABELA_2026)

# Recorte definido para o MVP
silver_base_2022 = bronze_2022.filter(F.col("CD_CARGO") == 7)
silver_base_2026 = bronze_2026.filter(F.col("CD_CARGO") == 7)

print("========== RECORTE SILVER — DEPUTADO ESTADUAL / SP ==========")
print("2022:", silver_base_2022.count(), "registros")
print("2026:", silver_base_2026.count(), "registros")

print("\nCargos identificados após o filtro:")
silver_base_2022.select("CD_CARGO", "DS_CARGO").distinct().show()
silver_base_2026.select("CD_CARGO", "DS_CARGO").distinct().show()

In [0]:
# ============================================================
# SILVER — Etapa 2: padronização das colunas essenciais
# ============================================================

def padronizar_candidatos(df):
    return (
        df.select(
            F.col("ANO_ELEICAO").alias("ano_eleicao"),

            # Mantém o valor original e cria uma data padronizada
            F.col("DT_ELEICAO").cast("string").alias("dt_eleicao_original"),
            F.expr("""
                COALESCE(
                    try_to_date(
                        TRIM(CAST(DT_ELEICAO AS STRING)),
                        'yyyy-MM-dd'
                    ),
                    try_to_date(
                        TRIM(CAST(DT_ELEICAO AS STRING)),
                        'dd/MM/yyyy'
                    )
                )
            """).alias("data_eleicao"),

            F.col("SG_UF").alias("uf"),
            F.col("CD_CARGO").alias("cd_cargo"),
            F.col("DS_CARGO").alias("cargo"),

            F.col("SQ_CANDIDATO").alias("sq_candidato"),
            F.col("NR_CANDIDATO").alias("nr_candidato"),
            F.col("NM_CANDIDATO").alias("nome_candidato"),
            F.col("NM_URNA_CANDIDATO").alias("nome_urna_candidato"),

            F.col("NR_CPF_CANDIDATO").alias("cpf_candidato"),
            F.when(F.col("NR_CPF_CANDIDATO") > 0, True)
             .otherwise(False)
             .alias("cpf_valido_para_cruzamento"),

            F.col("CD_GENERO").alias("cd_genero"),
            F.col("DS_GENERO").alias("genero"),

            F.col("CD_COR_RACA").alias("cd_cor_raca"),
            F.col("DS_COR_RACA").alias("cor_raca"),

            F.col("NR_PARTIDO").alias("nr_partido"),
            F.col("SG_PARTIDO").alias("sigla_partido"),
            F.col("NM_PARTIDO").alias("nome_partido"),

            F.col("CD_SITUACAO_CANDIDATURA").alias(
                "cd_situacao_candidatura"
            ),
            F.col("DS_SITUACAO_CANDIDATURA").alias(
                "situacao_candidatura"
            ),
            F.col("CD_SIT_TOT_TURNO").alias("cd_situacao_turno"),
            F.col("DS_SIT_TOT_TURNO").alias("situacao_turno")
        )
    )

silver_padronizada_2022 = padronizar_candidatos(silver_base_2022)
silver_padronizada_2026 = padronizar_candidatos(silver_base_2026)

print("========== VALIDAÇÃO DA PADRONIZAÇÃO ==========")

for ano, df in [
    (2022, silver_padronizada_2022),
    (2026, silver_padronizada_2026)
]:
    print(f"\n{ano}")
    print("Registros:", df.count())
    print("Data da eleição nula:", df.filter(
        F.col("data_eleicao").isNull()
    ).count())

    df.select(
        "ano_eleicao",
        "dt_eleicao_original",
        "data_eleicao",
        "cpf_valido_para_cruzamento"
    ).limit(5).show(truncate=False)

In [0]:
# ============================================================
# SILVER — Etapa 3: união 2022 x 2026 e controles de qualidade
# ============================================================

silver_candidatos_base = (
    silver_padronizada_2022
    .unionByName(silver_padronizada_2026)
)

print("========== CONTROLE DA BASE SILVER UNIFICADA ==========")
print("Total de registros:", silver_candidatos_base.count())

print("\nRegistros por ano:")
(
    silver_candidatos_base
    .groupBy("ano_eleicao")
    .count()
    .orderBy("ano_eleicao")
    .show()
)

print("\nCPF válido para cruzamento — contagem de registros:")
(
    silver_candidatos_base
    .groupBy(
        "ano_eleicao",
        "cpf_valido_para_cruzamento"
    )
    .count()
    .orderBy(
        "ano_eleicao",
        "cpf_valido_para_cruzamento"
    )
    .show()
)

print("\nSituação de candidatura preservada:")
(
    silver_candidatos_base
    .groupBy(
        "ano_eleicao",
        "situacao_candidatura"
    )
    .count()
    .orderBy(
        "ano_eleicao",
        "situacao_candidatura"
    )
    .show(truncate=False)
)

In [0]:
# ============================================================
# SILVER — Etapa 4: gravação da tabela de candidatos
# ============================================================

TABELA_SILVER = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "silver_candidatos_deputado_estadual_sp"
)

(
    silver_candidatos_base.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER)
)

print("Tabela Silver criada com sucesso:", TABELA_SILVER)

In [0]:
silver_gravada = spark.table(TABELA_SILVER)

print("========== CONTROLE DA TABELA SILVER ==========")
print("Registros:", silver_gravada.count())
print("Colunas:", len(silver_gravada.columns))

print("\nRegistros por ano:")
(
    silver_gravada
    .groupBy("ano_eleicao")
    .count()
    .orderBy("ano_eleicao")
    .show()
)

print("\nDatas padronizadas:")
(
    silver_gravada
    .groupBy("ano_eleicao", "data_eleicao")
    .count()
    .orderBy("ano_eleicao")
    .show()
)

display(
    silver_gravada.select(
        "ano_eleicao",
        "data_eleicao",
        "nome_candidato",
        "genero",
        "cor_raca",
        "sigla_partido",
        "situacao_candidatura",
        "cpf_valido_para_cruzamento"
    ).limit(10)
)

In [0]:
# ============================================================
# SILVER — Etapa 5: diagnóstico das fontes complementares
# ============================================================

TABELA_COMP_2022 = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "bronze_consulta_cand_complementar_2022"
)

TABELA_COMP_2026 = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "bronze_consulta_cand_complementar_2026"
)

complementar_2022 = spark.table(TABELA_COMP_2022)
complementar_2026 = spark.table(TABELA_COMP_2026)

for ano, df in [
    (2022, complementar_2022),
    (2026, complementar_2026)
]:
    print(f"\n========== COMPLEMENTAR {ano} ==========")
    print("Registros:", df.count())
    print("Colunas:", len(df.columns))
    print(
        "SQ_CANDIDATO nulo:",
        df.filter(F.col("SQ_CANDIDATO").isNull()).count()
    )
    print(
        "SQ_CANDIDATO distintos:",
        df.select("SQ_CANDIDATO").distinct().count()
    )

    print("\nColunas disponíveis:")
    print(", ".join(df.columns))

    display(df.limit(5))

In [0]:
# ============================================================
# SILVER — Etapa 6: validar a chave de junção complementar
# ============================================================

base_2022_ids = (
    silver_gravada
    .filter(F.col("ano_eleicao") == 2022)
    .select("sq_candidato")
    .distinct()
)

base_2026_ids = (
    silver_gravada
    .filter(F.col("ano_eleicao") == 2026)
    .select("sq_candidato")
    .distinct()
)

comp_2022_ids = (
    complementar_2022
    .select(F.col("SQ_CANDIDATO").alias("sq_candidato"))
    .distinct()
)

comp_2026_ids = (
    complementar_2026
    .select(F.col("SQ_CANDIDATO").alias("sq_candidato"))
    .distinct()
)

for ano, base_ids, comp_ids in [
    (2022, base_2022_ids, comp_2022_ids),
    (2026, base_2026_ids, comp_2026_ids)
]:
    print(f"\n========== VALIDAÇÃO DA JUNÇÃO — {ano} ==========")
    print("SQ_CANDIDATO distintos na Silver:", base_ids.count())
    print("SQ_CANDIDATO distintos no complementar:", comp_ids.count())
    print(
        "Candidatos Silver sem correspondente no complementar:",
        base_ids.join(comp_ids, "sq_candidato", "left_anti").count()
    )
    print(
        "Registros complementares fora do recorte Silver:",
        comp_ids.join(base_ids, "sq_candidato", "left_anti").count()
    )

duplicados_comp_2022 = (
    complementar_2022
    .groupBy("SQ_CANDIDATO")
    .count()
    .filter(F.col("count") > 1)
)

print("\n========== DUPLICIDADES NO COMPLEMENTAR 2022 ==========")

display(
    complementar_2022
    .join(duplicados_comp_2022, "SQ_CANDIDATO", "inner")
    .select(
        "SQ_CANDIDATO",
        "DT_GERACAO",
        "HH_GERACAO",
        "CD_DETALHE_SITUACAO_CAND",
        "DS_DETALHE_SITUACAO_CAND",
        "CD_SITUACAO_CANDIDATO_TOT",
        "DS_SITUACAO_CANDIDATO_TOT"
    )
    .orderBy("SQ_CANDIDATO", "DT_GERACAO", "HH_GERACAO")
)

In [0]:
# ============================================================
# SILVER — Etapa 6: validar a chave de junção complementar
# ============================================================

base_2022_ids = (
    silver_gravada
    .filter(F.col("ano_eleicao") == 2022)
    .select("sq_candidato")
    .distinct()
)

base_2026_ids = (
    silver_gravada
    .filter(F.col("ano_eleicao") == 2026)
    .select("sq_candidato")
    .distinct()
)

comp_2022_ids = (
    complementar_2022
    .select(F.col("SQ_CANDIDATO").alias("sq_candidato"))
    .distinct()
)

comp_2026_ids = (
    complementar_2026
    .select(F.col("SQ_CANDIDATO").alias("sq_candidato"))
    .distinct()
)

for ano, base_ids, comp_ids in [
    (2022, base_2022_ids, comp_2022_ids),
    (2026, base_2026_ids, comp_2026_ids)
]:
    print(f"\n========== VALIDAÇÃO DA JUNÇÃO — {ano} ==========")

    print(
        "SQ_CANDIDATO distintos na Silver:",
        base_ids.count()
    )

    print(
        "SQ_CANDIDATO distintos no complementar:",
        comp_ids.count()
    )

    print(
        "Candidatos Silver sem correspondente no complementar:",
        base_ids.join(
            comp_ids,
            "sq_candidato",
            "left_anti"
        ).count()
    )

    print(
        "Registros complementares fora do recorte Silver:",
        comp_ids.join(
            base_ids,
            "sq_candidato",
            "left_anti"
        ).count()
    )

duplicados_comp_2022 = (
    complementar_2022
    .groupBy("SQ_CANDIDATO")
    .count()
    .filter(F.col("count") > 1)
)

print("\n========== DUPLICIDADES NO COMPLEMENTAR 2022 ==========")

display(
    complementar_2022
    .join(
        duplicados_comp_2022,
        "SQ_CANDIDATO",
        "inner"
    )
    .select(
        "SQ_CANDIDATO",
        "DT_GERACAO",
        "HH_GERACAO",
        "CD_DETALHE_SITUACAO_CAND",
        "DS_DETALHE_SITUACAO_CAND",
        "CD_SITUACAO_CANDIDATO_TOT",
        "DS_SITUACAO_CANDIDATO_TOT"
    )
    .orderBy(
        "SQ_CANDIDATO",
        "DT_GERACAO",
        "HH_GERACAO"
    )
)

In [0]:
# ============================================================
# SILVER — Etapa 7: verificar se as duplicidades 2022
# são registros completamente idênticos
# ============================================================

ids_duplicados_2022 = (
    complementar_2022
    .groupBy("SQ_CANDIDATO")
    .count()
    .filter(F.col("count") > 1)
    .select("SQ_CANDIDATO")
)

registros_duplicados_2022 = (
    complementar_2022
    .join(
        ids_duplicados_2022,
        on="SQ_CANDIDATO",
        how="inner"
    )
)

print("========== ANÁLISE DAS DUPLICIDADES 2022 ==========")

print(
    "SQ_CANDIDATO duplicados:",
    ids_duplicados_2022.count()
)

print(
    "Total de linhas desses candidatos:",
    registros_duplicados_2022.count()
)

print(
    "Linhas distintas considerando TODAS as 49 colunas:",
    registros_duplicados_2022.distinct().count()
)

display(
    registros_duplicados_2022
    .orderBy("SQ_CANDIDATO")
)

In [0]:
# ============================================================
# SILVER — Etapa 8:
# validar CD_ELEICAO dos candidatos duplicados
# na fonte principal de 2022
# ============================================================

ids_duplicados_lista = [
    250001612464,
    250001612465,
    250001615967,
    250001615968
]

principal_2022 = spark.table(
    "mvp_engenharia_de_dados_puc_rio.dados_tse.bronze_consulta_cand_2022"
)

display(
    principal_2022
    .filter(
        F.col("SQ_CANDIDATO").isin(ids_duplicados_lista)
    )
    .select(
        "SQ_CANDIDATO",
        "CD_ELEICAO",
        "DS_ELEICAO",
        "CD_CARGO",
        "DS_CARGO",
        "NR_CANDIDATO",
        "NM_CANDIDATO"
    )
    .orderBy(
        "SQ_CANDIDATO",
        "CD_ELEICAO"
    )
)

In [0]:
# ============================================================
# SILVER — Etapa 9: recortar as fontes complementares
# somente para candidatos presentes na Silver
# ============================================================

comp_2022_recorte = (
    complementar_2022
    .select(
        F.col("ANO_ELEICAO").alias("ano_eleicao"),
        F.col("SQ_CANDIDATO").alias("sq_candidato"),
        F.col("DS_DETALHE_SITUACAO_CAND").alias("detalhe_situacao_candidatura"),
        F.col("DS_NACIONALIDADE").alias("nacionalidade"),
        F.col("NM_MUNICIPIO_NASCIMENTO").alias("municipio_nascimento"),
        F.col("NR_IDADE_DATA_POSSE").alias("idade_data_posse"),
        F.col("ST_QUILOMBOLA").alias("st_quilombola"),
        F.col("DS_ETNIA_INDIGENA").alias("etnia_indigena"),
        F.col("VR_DESPESA_MAX_CAMPANHA").alias("limite_despesa_campanha"),
        F.col("ST_REELEICAO").alias("st_reeleicao"),
        F.col("ST_DECLARAR_BENS").alias("st_declarar_bens")
    )
    .join(
        base_2022_ids.withColumn("ano_eleicao", F.lit(2022)),
        on=["ano_eleicao", "sq_candidato"],
        how="inner"
    )
)

comp_2026_recorte = (
    complementar_2026
    .select(
        F.col("ANO_ELEICAO").alias("ano_eleicao"),
        F.col("SQ_CANDIDATO").alias("sq_candidato"),
        F.col("DS_DETALHE_SITUACAO_CAND").alias("detalhe_situacao_candidatura"),
        F.col("DS_NACIONALIDADE").alias("nacionalidade"),
        F.col("NM_MUNICIPIO_NASCIMENTO").alias("municipio_nascimento"),
        F.col("NR_IDADE_DATA_POSSE").alias("idade_data_posse"),
        F.col("ST_QUILOMBOLA").alias("st_quilombola"),
        F.col("DS_ETNIA_INDIGENA").alias("etnia_indigena"),
        F.col("VR_DESPESA_MAX_CAMPANHA").alias("limite_despesa_campanha"),
        F.col("ST_REELEICAO").alias("st_reeleicao"),
        F.col("ST_DECLARAR_BENS").alias("st_declarar_bens")
    )
    .join(
        base_2026_ids.withColumn("ano_eleicao", F.lit(2026)),
        on=["ano_eleicao", "sq_candidato"],
        how="inner"
    )
)

print("========== CONTROLE DO RECORTE COMPLEMENTAR ==========")

print(
    "2022:",
    comp_2022_recorte.count(),
    "registros |",
    comp_2022_recorte.select("sq_candidato").distinct().count(),
    "SQ_CANDIDATO distintos"
)

print(
    "2026:",
    comp_2026_recorte.count(),
    "registros |",
    comp_2026_recorte.select("sq_candidato").distinct().count(),
    "SQ_CANDIDATO distintos"
)

In [0]:
# ============================================================
# SILVER — Etapa 10: enriquecer a Silver com dados complementares
# ============================================================

complementar_silver = (
    comp_2022_recorte
    .unionByName(comp_2026_recorte)
    .withColumn(
        "registro_complementar_encontrado",
        F.lit(True)
    )
)

print("========== CONTROLE COMPLEMENTAR UNIFICADO ==========")
print("Registros:", complementar_silver.count())
print(
    "Chaves distintas:",
    complementar_silver
    .select("ano_eleicao", "sq_candidato")
    .distinct()
    .count()
)

# Enriquecimento da Silver
silver_enriquecida = (
    silver_gravada
    .join(
        complementar_silver,
        on=["ano_eleicao", "sq_candidato"],
        how="left"
    )
)

print("\n========== CONTROLE SILVER ENRIQUECIDA ==========")
print("Registros:", silver_enriquecida.count())
print("Colunas:", len(silver_enriquecida.columns))

print(
    "Sem correspondente complementar:",
    silver_enriquecida
    .filter(
        F.col("registro_complementar_encontrado").isNull()
    )
    .count()
)

print("\nRegistros por ano:")
(
    silver_enriquecida
    .groupBy("ano_eleicao")
    .count()
    .orderBy("ano_eleicao")
    .show()
)

display(
    silver_enriquecida.select(
        "ano_eleicao",
        "sq_candidato",
        "nome_candidato",
        "genero",
        "cor_raca",
        "sigla_partido",
        "situacao_candidatura",
        "detalhe_situacao_candidatura",
        "nacionalidade",
        "municipio_nascimento",
        "idade_data_posse",
        "st_quilombola",
        "etnia_indigena",
        "limite_despesa_campanha",
        "st_reeleicao",
        "st_declarar_bens"
    ).limit(20)
)

In [0]:
# ============================================================
# SILVER — Etapa 11: gravar Silver enriquecida definitiva
# ============================================================

silver_final = (
    silver_enriquecida
    .drop("registro_complementar_encontrado")
)

TABELA_SILVER = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "silver_candidatos_deputado_estadual_sp"
)

TABELA_SILVER_TMP = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "silver_candidatos_deputado_estadual_sp_tmp"
)

# 1. Grava uma tabela temporária
(
    silver_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER_TMP)
)

# 2. Substitui a Silver oficial
spark.sql(f"""
CREATE OR REPLACE TABLE {TABELA_SILVER}
USING DELTA
AS
SELECT *
FROM {TABELA_SILVER_TMP}
""")

# 3. Remove a temporária
spark.sql(
    f"DROP TABLE IF EXISTS {TABELA_SILVER_TMP}"
)

print("Tabela Silver enriquecida gravada com sucesso.")

In [0]:
silver_gravada = spark.table(TABELA_SILVER)

print("========== CONTROLE FINAL DA SILVER ==========")

print("Registros:", silver_gravada.count())
print("Colunas:", len(silver_gravada.columns))

print(
    "Chaves distintas:",
    silver_gravada
    .select("ano_eleicao", "sq_candidato")
    .distinct()
    .count()
)

print("\nRegistros por ano:")

(
    silver_gravada
    .groupBy("ano_eleicao")
    .count()
    .orderBy("ano_eleicao")
    .show()
)

In [0]:
# ============================================================
# SILVER — Etapa 12: Data Quality
# ============================================================

print("========== DATA QUALITY — SILVER ==========")

total = silver_gravada.count()

print("\n1. UNICIDADE DA CHAVE LÓGICA")

duplicados_chave = (
    silver_gravada
    .groupBy("ano_eleicao", "sq_candidato")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total de registros:", total)
print("Chaves duplicadas:", duplicados_chave)


print("\n2. COMPLETUDE — CAMPOS CRÍTICOS")

campos_criticos = [
    "ano_eleicao",
    "sq_candidato",
    "nome_candidato",
    "data_eleicao",
    "genero",
    "cor_raca",
    "sigla_partido"
]

for campo in campos_criticos:
    qtd_nulos = (
        silver_gravada
        .filter(F.col(campo).isNull())
        .count()
    )

    print(
        f"{campo}: {qtd_nulos} nulos "
        f"({round(qtd_nulos / total * 100, 2)}%)"
    )


print("\n3. CPF VÁLIDO PARA CRUZAMENTO")

(
    silver_gravada
    .groupBy(
        "ano_eleicao",
        "cpf_valido_para_cruzamento"
    )
    .count()
    .orderBy(
        "ano_eleicao",
        "cpf_valido_para_cruzamento"
    )
    .show()
)


print("\n4. DOMÍNIO — GÊNERO")

(
    silver_gravada
    .groupBy("ano_eleicao", "genero")
    .count()
    .orderBy("ano_eleicao", "genero")
    .show()
)


print("\n5. DOMÍNIO — COR/RAÇA")

(
    silver_gravada
    .groupBy("ano_eleicao", "cor_raca")
    .count()
    .orderBy("ano_eleicao", "cor_raca")
    .show()
)


print("\n6. SITUAÇÃO DA CANDIDATURA")

(
    silver_gravada
    .groupBy(
        "ano_eleicao",
        "situacao_candidatura"
    )
    .count()
    .orderBy(
        "ano_eleicao",
        "situacao_candidatura"
    )
    .show()
)


print("\n7. CONSISTÊNCIA DA DATA DA ELEIÇÃO")

(
    silver_gravada
    .groupBy(
        "ano_eleicao",
        "data_eleicao"
    )
    .count()
    .orderBy("ano_eleicao")
    .show()
)